In [16]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score
import os

In [17]:
# Constants
RandomSTATE = 1082
TRAIN_SIZE = 0.2
MODEL_NAME = "StackingClassifier(LogisticRegression RandomForest SVC)"
DATASET_NAME = "GPS_Data_Simplified_2D_Feature_Map.xlsx"
RESULTS_CSV = "../../../2D_Results.csv"

In [18]:
# Load the dataset
try:
    df = pd.read_excel("../../../Datasets/excel/2dSpoofingDataset/GPS_Data_Simplified_2D_Feature_Map.xlsx")
except FileNotFoundError:
    print("Error: The file '../Datasets/excel/GPS_Data_Simplified_2D_Feature_Map.xlsx' was not found.")
    print("Please make sure the path is correct.")
    exit()

In [19]:
# Data Preprocessing

X = df.drop('Output', axis=1)
y = df['Output']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, train_size=TRAIN_SIZE, random_state=RandomSTATE, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [20]:
print("Number of classes:", len(set(y)))
print("Class distribution:\n", y.value_counts())

Number of classes: 4
Class distribution:
 Output
0    397825
2     44232
1     36458
3     32015
Name: count, dtype: int64


In [21]:
# Define base models
base_learners = [
    ('lr', LogisticRegression(random_state=RandomSTATE, max_iter=1200, solver='saga')),
    ('rf', RandomForestClassifier(random_state=RandomSTATE)),
    ('svc', SVC(probability=True, random_state=RandomSTATE))
]

# Meta model
meta_learner = LogisticRegression(random_state=RandomSTATE, max_iter=1200, solver='saga')

In [22]:
# Create and train the Stacking model
stacking_model = StackingClassifier(
    estimators=base_learners,
    final_estimator=meta_learner,
    cv=5,
    passthrough=False,
    n_jobs=-1
)
stacking_model.fit(X_train_scaled, y_train)

StackingClassifier(cv=5,
                   estimators=[('lr',
                                LogisticRegression(max_iter=1200,
                                                   random_state=1082,
                                                   solver='saga')),
                               ('rf',
                                RandomForestClassifier(random_state=1082)),
                               ('svc',
                                SVC(probability=True, random_state=1082))],
                   final_estimator=LogisticRegression(max_iter=1200,
                                                      random_state=1082,
                                                      solver='saga'),
                   n_jobs=-1)

In [23]:
# Predict and evaluate
y_pred = stacking_model.predict(X_test_scaled)
accuracy = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred, average='weighted', zero_division=0)
print(f"Stacked Model Accuracy on Test Set: {accuracy * 100:.2f}%")
print(f"F1 Score: {f1:.4f}")
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred, zero_division=0))

Stacked Model Accuracy on Test Set: 92.88%
F1 Score: 0.9272
Confusion Matrix:
[[309414   2283   5592    971]
 [  3425  23081   2660      0]
 [  9439   3029  22918      0]
 [  1679      3      0  23930]]

Classification Report:
              precision    recall  f1-score   support

           0       0.96      0.97      0.96    318260
           1       0.81      0.79      0.80     29166
           2       0.74      0.65      0.69     35386
           3       0.96      0.93      0.95     25612

    accuracy                           0.93    408424
   macro avg       0.87      0.84      0.85    408424
weighted avg       0.93      0.93      0.93    408424



In [24]:
# Save results to CSV
results = [{
    "model": MODEL_NAME,
    "dataset": DATASET_NAME,
    "f1score": f1,
    "accuracy": accuracy
}]
results_df = pd.DataFrame(results)

file_exists = os.path.isfile(RESULTS_CSV)
results_df.to_csv(
    RESULTS_CSV,
    mode='a',
    header=not file_exists,
    index=False
)
print(f"Results appended to {RESULTS_CSV}")

Results appended to ../../../2D_Results.csv
